# Argus — Tiny grid detector (YOLO-style CNN, built from scratch)

Our own **single-shot object detector**, written layer by layer in Keras without Ultralytics.
It finds **eyes (open/closed), a yawning mouth and the head position** in a face crop. It's one
fully convolutional network that mixes **classification** (objectness + class per grid cell)
with **regression** (box centre and size per grid cell), the idea behind YOLO.

**Where it sits in Argus:**

    camera frame → BlazeFace face crop → THIS DETECTOR → per-frame eye / yawn / head state
                                                       → (later) LSTM over the 5 fps window → Not Drowsy / Drowsy

It must run on a Raspberry Pi 5 CPU, so the targets are **≤ ~100k parameters, int8 TFLite,
grayscale input**.

**Two steps, both trained and reported:**
1. **YOLOv1-style baseline.** A plain conv stack, an S×S grid, one box per cell, and the original
   sum-squared YOLO loss. It's the textbook version, kept as the reference.
2. **Modern tiny YOLO.** CSP/C2f-lite blocks, an SPPF-lite block, a decoupled head, an
   anchor-free head with neighbour-cell assignment, and GIoU + BCE losses. It's kept only if it
   beats step 1 on validation mAP.

**Reads:** `dataset/parts_det.zip`, built by `src/dataset/scripts/build_detection_dataset.py`.
It holds YOLO labels for `open_eye`, `closed_eye`, `yawn`, `head_straight` and `head_down` on
192 px face crops, plus MRL and Roboflow close-ups.

**Writes:**
- `models/parts_det/tiny_yolo_*.keras` and `*_int8.tflite`;
- `models/parts_det/<name>_metrics.json`, read by notebook 14's comparison table.

**Compared against:** `13_yolo_nano.ipynb` (Ultralytics YOLO11n / YOLOv5n) and
`14_backbone_grid_detector.ipynb` (MobileNetV3-Small + this notebook's head). All three share the
load, evaluation and latency cells verbatim.

## Setup

**Getting the dataset in (Colab).** Upload `parts_det.zip`, built by
`src/dataset/scripts/build_detection_dataset.py`, **straight into the session**. Either drag it
into the Files panel (left sidebar → 📁 → upload, so it lands at `/content/parts_det.zip`), or
run the Loading cell and use the upload button it shows if the zip isn't found. Drive isn't
needed for the dataset.

**Keeping the results.** A Colab session's disk is wiped when it ends.
- With `USE_DRIVE = True` (the default), Drive is mounted only to save trained models and
  metrics to `MyDrive/Argus/models/parts_det/`. They survive the session, and notebook 14's
  comparison table can read all three notebooks' results.
- With `USE_DRIVE = False`, everything is saved under `/content/models/parts_det/` and the last
  cell zips it for download.

**Locally:** set `ARGUS_DET_ZIP`, and optionally `ARGUS_PROJECT_FOLDER` / `ARGUS_WORK_DIR`.
`ARGUS_SMOKE=1` turns every training run into a few-step smoke test on a small subset. It's used
to run the notebook end to end on a CPU before spending GPU time.

**This cell is shared verbatim by notebooks 12 / 13 / 14** so their numbers are comparable;
change it in all three or none.

In [ ]:
import os
import sys
import json
import time
import shutil
import zipfile
import datetime

import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("ARGUS_SMOKE", "0") == "1"
USE_DRIVE = True          # Colab only: save models/metrics to Drive so they outlive the session

if IN_COLAB:
    WORK_DIR = "/content"
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        project_folder = "/content/drive/MyDrive/Argus"
    else:
        project_folder = "/content"
else:
    project_folder = os.environ.get("ARGUS_PROJECT_FOLDER", os.path.expanduser("~/argus-colab-mirror"))
    WORK_DIR = os.environ.get("ARGUS_WORK_DIR", "/tmp/argus_work")

models_folder = f"{project_folder}/models/parts_det"
# Where the dataset zip may be, in order: explicit env var, uploaded into the session, on Drive.
DET_ZIP_CANDIDATES = [p for p in (os.environ.get("ARGUS_DET_ZIP"), f"{WORK_DIR}/parts_det.zip",
                                  f"{project_folder}/dataset/parts_det.zip") if p]
os.makedirs(models_folder, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)
VERSION_STR = datetime.datetime.now().strftime("%Y%m%d_%H%M")

# Dataset contract (src/dataset/argus_dataset/config.py: DET_CLASSES, DET_IMG_SIZE = 192 stored).
CLASSES = ["open_eye", "closed_eye", "yawn", "head_straight", "head_down"]
NUM_CLASSES = len(CLASSES)
EYE_CLASSES, YAWN_CLASS, HEAD_CLASSES = (0, 1), 2, (3, 4)

IMG = 160                 # model input side (stored crops are 192 px, resized here)
VARIANT = "mix"           # "gray" | "ir" | "mix" (train: random gray/IR blend per image)
EVAL_VARIANT = "ir"       # what val/test images look like: the cabin camera is a Pi NoIR
INCLUDE_CLOSEUPS = True   # MRL + Roboflow eye/mouth close-ups (whole image = one box)
INCLUDE_PSEUDO_EYES = True  # images whose skipped eyes were pseudo-labelled from landmarks
CONF_THR = 0.25           # operating point for P/R, eye-state and head accuracy
SEED = 42
np.random.seed(SEED)
print(f"IN_COLAB={IN_COLAB} SMOKE={SMOKE} USE_DRIVE={USE_DRIVE}\nmodels -> {models_folder}")

## Loading the detection dataset

The zip is found in this order:
1. `ARGUS_DET_ZIP`;
2. `/content/parts_det.zip`, i.e. uploaded into the session;
3. `MyDrive/Argus/dataset/parts_det.zip`;
4. otherwise an upload button.

It's unzipped to local session disk once. Training reads the local files, never Drive's FUSE
mount: thousands of small reads off Drive time out, as `07`/`11` found.

The zip is YOLO format:
- `gray/` and `ir/`, each with `{train,valid,test}/{images,labels}`;
- `index.csv`, one row per source image with `status`, `kind`, `source`, `group_id` and `split`.

**Splits are by group:** an original image plus its augmented copies and near-duplicates, or one
MRL subject. They were assigned when the dataset was built. Never re-split by image here.

Each label line is `class cx cy w h`, normalised to 0–1.

**This cell is shared verbatim by notebooks 12 / 13 / 14.**

In [ ]:
LOCAL_DIR = f"{WORK_DIR}/parts_det"


def find_dataset_zip():
    for p in DET_ZIP_CANDIDATES:
        if os.path.isfile(p) and zipfile.is_zipfile(p):
            return p
    if IN_COLAB:   # not uploaded yet: show an upload button (Files panel drag-and-drop is faster for 371 MB)
        from google.colab import files
        print("parts_det.zip not found — choose it in the upload button below.")
        for name in files.upload():
            if name.endswith(".zip"):
                dest = f"{WORK_DIR}/parts_det.zip"
                if os.path.abspath(name) != dest:
                    shutil.move(name, dest)
                return dest
    raise FileNotFoundError(f"parts_det.zip not found in {DET_ZIP_CANDIDATES}")


if not os.path.isfile(f"{LOCAL_DIR}/index.csv"):
    DET_ZIP = find_dataset_zip()
    print(f"Extracting {DET_ZIP} -> {LOCAL_DIR}")
    with zipfile.ZipFile(DET_ZIP) as zf:
        zf.extractall(LOCAL_DIR)

index = pd.read_csv(f"{LOCAL_DIR}/index.csv", dtype=str, keep_default_na=False)
ok = index[index.status == "ok"].copy()
if not INCLUDE_CLOSEUPS:
    ok = ok[ok.kind != "closeup"]
if not INCLUDE_PSEUDO_EYES:
    ok = ok[ok.pseudo_eyes.isin(["", "0"])]


def read_labels(label_rel):
    """YOLO label file -> (classes int32 (n,), boxes float32 (n,4) as cx,cy,w,h)."""
    rows = [l.split() for l in open(f"{LOCAL_DIR}/{label_rel}").read().splitlines() if l.strip()]
    if not rows:
        return np.zeros(0, np.int32), np.zeros((0, 4), np.float32)
    a = np.array(rows, np.float32)
    return a[:, 0].astype(np.int32), a[:, 1:5]


splits = {}
for s in ("train", "valid", "test"):
    df = ok[ok.split == s].reset_index(drop=True)
    if SMOKE:   # a few images of every (source, kind) so every code path still runs
        df = (df.groupby(["source", "kind"], group_keys=False)
                .apply(lambda g: g.sample(min(len(g), 24), random_state=SEED)).reset_index(drop=True))
    labels = [read_labels(p) for p in df.label_path]
    df["classes"] = [c for c, _ in labels]
    df["boxes"] = [b for _, b in labels]
    splits[s] = df

summary = pd.DataFrame({
    s: {"images": len(d), **{c: int(sum(int((a == i).sum()) for a in d.classes))
                             for i, c in enumerate(CLASSES)}}
    for s, d in splits.items()})
print(summary)


def image_path(row, variant):
    return f"{LOCAL_DIR}/{row['gray_path'] if variant == 'gray' else row['ir_path']}"


def gts_for(df):
    """Ground truth per image as xyxy boxes, for the shared evaluation."""
    out = {}
    for r in df.itertuples():
        b = r.boxes
        out[r.image_id] = (np.c_[b[:, 0] - b[:, 2] / 2, b[:, 1] - b[:, 3] / 2,
                                 b[:, 0] + b[:, 2] / 2, b[:, 1] + b[:, 3] / 2], r.classes)
    return out

## Shared evaluation

Each model's output is converted to a common form: per image, `xyxy` boxes (0–1), scores and
class ids, after NMS. It is then scored here, so 12, 13 and 14 report **identically defined**
numbers.

- **mAP@0.5 and mAP@0.5:0.95** use COCO-style 101-point interpolation, per class.
- **P / R per class** are taken at `CONF_THR` and IoU ≥ 0.5.
- **Eye-state accuracy** is what the temporal stage will actually consume. Each ground-truth eye
  is matched to the best-IoU predicted eye box of either eye class (IoU ≥ 0.5, score ≥
  `CONF_THR`). From that we get:
  - `eye_recall`: the share of eyes found;
  - `eye_state_acc`: the open/closed correctness among found eyes;
  - `eye_state_overall`: correct over all eyes.
- **Head accuracy** is defined the same way over `head_straight` / `head_down`.

`report()` also splits these by `kind` (face / closeup) and `source` (roboflow / mrl_eye),
because MRL close-ups are easy and must not hide weak face-crop results.

**Shared verbatim by notebooks 12 / 13 / 14.**

In [ ]:
def iou_xyxy(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), np.float32)
    x0 = np.maximum(a[:, None, 0], b[None, :, 0]); y0 = np.maximum(a[:, None, 1], b[None, :, 1])
    x1 = np.minimum(a[:, None, 2], b[None, :, 2]); y1 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x1 - x0, 0, None) * np.clip(y1 - y0, 0, None)
    area = lambda r: (r[:, 2] - r[:, 0]) * (r[:, 3] - r[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)


def nms_per_class(boxes, scores, classes, iou_thr=0.5, max_det=30):
    keep = []
    for c in np.unique(classes):
        idx = np.where(classes == c)[0]
        idx = idx[np.argsort(-scores[idx])]
        while len(idx):
            i = idx[0]
            keep.append(i)
            if len(idx) == 1:
                break
            idx = idx[1:][iou_xyxy(boxes[i:i + 1], boxes[idx[1:]])[0] < iou_thr]
    keep = np.array(sorted(keep, key=lambda i: -scores[i])[:max_det], np.int64)
    return boxes[keep], scores[keep], classes[keep]


def _ap(recall, precision):
    r = np.concatenate([[0.0], recall, [1.0]]); p = np.concatenate([[1.0], precision, [0.0]])
    p = np.flip(np.maximum.accumulate(np.flip(p)))
    return float(np.mean(np.interp(np.linspace(0, 1, 101), r, p)))


def average_precision(preds, gts, cls, iou_thrs):
    """AP of one class at each IoU threshold over a set of images."""
    items, n_gt = [], 0
    for iid, (gb, gc) in gts.items():
        g = gb[gc == cls]; n_gt += len(g)
        pb, ps, pc = preds.get(iid, (np.zeros((0, 4)), np.zeros(0), np.zeros(0, int)))
        m = pc == cls
        items.append((g, pb[m], ps[m]))
    if n_gt == 0:
        return [float("nan")] * len(iou_thrs)
    out = []
    for t in iou_thrs:
        scores, tps = [], []
        for g, pb, ps in items:
            order = np.argsort(-ps); pb, ps = pb[order], ps[order]
            ious = iou_xyxy(pb, g); used = np.zeros(len(g), bool)
            for k in range(len(pb)):
                j = int(np.argmax(ious[k])) if len(g) else -1
                hit = j >= 0 and ious[k, j] >= t and not used[j]
                if hit:
                    used[j] = True
                scores.append(ps[k]); tps.append(hit)
        if not scores:
            out.append(0.0); continue
        order = np.argsort(-np.array(scores)); tp = np.array(tps)[order]
        ctp, cfp = np.cumsum(tp), np.cumsum(~tp)
        out.append(_ap(ctp / n_gt, ctp / np.maximum(ctp + cfp, 1)))
    return out


def state_accuracy(preds, gts, group, conf=CONF_THR, iou_thr=0.5):
    """For GT boxes whose class is in `group`: found-rate and state accuracy (see markdown)."""
    total = found = correct = 0
    for iid, (gb, gc) in gts.items():
        m = np.isin(gc, group)
        if not m.any():
            continue
        pb, ps, pc = preds.get(iid, (np.zeros((0, 4)), np.zeros(0), np.zeros(0, int)))
        k = np.isin(pc, group) & (ps >= conf)
        ious = iou_xyxy(gb[m], pb[k])
        for g_i, cls in enumerate(gc[m]):
            total += 1
            if ious.shape[1] and ious[g_i].max() >= iou_thr:
                found += 1
                correct += int(pc[k][int(np.argmax(ious[g_i]))] == cls)
    return {"n": total, "recall": found / max(total, 1), "state_acc": correct / max(found, 1),
            "state_overall": correct / max(total, 1)}


def pr_at_conf(preds, gts, cls, conf=CONF_THR, iou_thr=0.5):
    tp = fp = n_gt = 0
    for iid, (gb, gc) in gts.items():
        g = gb[gc == cls]; n_gt += len(g)
        pb, ps, pc = preds.get(iid, (np.zeros((0, 4)), np.zeros(0), np.zeros(0, int)))
        p = pb[(pc == cls) & (ps >= conf)]
        ious = iou_xyxy(p, g); used = np.zeros(len(g), bool)
        for k in range(len(p)):
            j = int(np.argmax(ious[k])) if len(g) else -1
            if j >= 0 and ious[k, j] >= iou_thr and not used[j]:
                used[j] = True; tp += 1
            else:
                fp += 1
    return tp / max(tp + fp, 1), tp / max(n_gt, 1)


def evaluate(preds, gts):
    thrs = np.arange(0.5, 0.96, 0.05)
    aps = {c: average_precision(preds, gts, i, thrs) for i, c in enumerate(CLASSES)}
    res = {
        "map50": float(np.nanmean([a[0] for a in aps.values()])),
        "map50_95": float(np.nanmean([np.mean(a) for a in aps.values()])),
        "ap50": {c: a[0] for c, a in aps.items()},
        "pr_at_conf": {c: dict(zip(("precision", "recall"), pr_at_conf(preds, gts, i)))
                       for i, c in enumerate(CLASSES)},
        "eye": state_accuracy(preds, gts, EYE_CLASSES),
        "head": state_accuracy(preds, gts, HEAD_CLASSES),
    }
    return res


def report(preds, df, title=""):
    """Overall + per-kind / per-source evaluation of `preds` on the images of `df`."""
    out = {"all": evaluate(preds, gts_for(df))}
    for col in ("kind", "source"):
        for v, sub in df.groupby(col):
            out[f"{col}={v}"] = evaluate(preds, gts_for(sub))
    rows = {k: {"mAP50": r["map50"], "mAP50-95": r["map50_95"],
                **{f"AP50 {c}": r["ap50"][c] for c in CLASSES},
                "eye recall": r["eye"]["recall"], "eye state acc": r["eye"]["state_acc"],
                "head recall": r["head"]["recall"], "head acc": r["head"]["state_acc"]}
            for k, r in out.items()}
    print(f"== {title}")
    print(pd.DataFrame(rows).T.round(3).to_string())
    return out

## Shared latency benchmark and metrics file

The Pi runs **int8 TFLite**. `bench_tflite` times one image through a `.tflite` model with
`num_threads` set to 1, 2 and 4: the Pi 5 has 4 cores, and small models often don't speed up
with more threads, which is the MobileNet-on-Pi question. Colab's CPU is only a *proxy*. The
saved `.tflite` is what gets timed on the real Pi with `benchmark_model`.

`save_metrics` writes `models/parts_det/<name>_metrics.json`. Notebook 14's last cell reads
every such file into one comparison table.

**Shared verbatim by notebooks 12 / 13 / 14.**

In [ ]:
import tensorflow as tf


def bench_tflite(path, threads=(1, 2, 4), runs=200, warmup=20):
    out = {}
    for n in threads:
        it = tf.lite.Interpreter(model_path=str(path), num_threads=n)
        it.allocate_tensors()
        inp = it.get_input_details()[0]
        x = np.random.rand(*inp["shape"]).astype(np.float32)
        if inp["dtype"] != np.float32:   # integer-input model: feed in its quantised domain
            s, z = inp["quantization"]
            x = np.clip(np.round(x / s + z), np.iinfo(inp["dtype"]).min, np.iinfo(inp["dtype"]).max).astype(inp["dtype"])
        ts = []
        for k in range(warmup + (5 if SMOKE else runs)):
            t0 = time.perf_counter()
            it.set_tensor(inp["index"], x); it.invoke()
            if k >= warmup:
                ts.append((time.perf_counter() - t0) * 1000)
        out[str(n)] = round(float(np.median(ts)), 3)
    print(f"latency (median ms) {os.path.basename(str(path))}: {out}")
    return out


def _jsonable(o):
    if isinstance(o, dict):
        return {k: _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_jsonable(v) for v in o]
    if isinstance(o, (np.floating, float)):
        return None if np.isnan(o) else float(o)
    if isinstance(o, np.integer):
        return int(o)
    return o


def save_metrics(name, notebook, **fields):
    rec = {"name": name, "notebook": notebook, "variant": VARIANT, "eval_variant": EVAL_VARIANT,
           "img": IMG, "smoke": SMOKE, "version": VERSION_STR, **fields}
    path = f"{models_folder}/{name}_metrics.json"
    with open(path, "w") as f:
        json.dump(_jsonable(rec), f, indent=2)
    print(f"-> {path}")
    return path

## `tf.data` pipeline with box-aware augmentation

Images are decoded as **one-channel grayscale** (what the Pi NoIR camera gives after conversion)
and resized to `IMG`. Pixel scaling to 0–1 happens here, so a model sees 0–1 input.

**Training augmentation** keeps every box consistent with the image:
- horizontal flip (the classes have no left/right);
- random zoom/translate, 80–100% crops, where a box whose centre leaves the frame is dropped;
- brightness, contrast, gamma and sensor noise.

With `VARIANT = "mix"`, each training image is a random blend of its plain-gray and IR-simulated
versions. That covers both the daytime NoIR look and the night IR-LED look. Validation and test
images use `EVAL_VARIANT`.

Targets are built by an `encoder` (one per detector family, below) through `tf.numpy_function`,
so the grid-assignment logic stays plain NumPy and easy to read.

**Shared verbatim by notebooks 12 / 14.**

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
MAX_BOXES = 10


def pack(df):
    n = len(df)
    B = np.zeros((n, MAX_BOXES, 4), np.float32)
    C = -np.ones((n, MAX_BOXES), np.int32)
    for k, (c, b) in enumerate(zip(df.classes, df.boxes)):
        m = min(len(c), MAX_BOXES)
        B[k, :m], C[k, :m] = b[:m], c[:m]
    gray = [image_path(r, "gray") for _, r in df.iterrows()]
    ir = [image_path(r, "ir") for _, r in df.iterrows()]
    return gray, ir, B, C


def load_image(path):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=1)
    return tf.image.resize(tf.cast(img, tf.float32) / 255.0, (IMG, IMG), antialias=True)


def augment(img, boxes, cls):
    flip = tf.random.uniform(()) < 0.5
    img = tf.cond(flip, lambda: tf.image.flip_left_right(img), lambda: img)
    boxes = tf.where(flip, tf.stack([1.0 - boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]], -1), boxes)

    s = tf.random.uniform((), 0.8, 1.0)
    u = tf.random.uniform((), 0.0, 1.0 - s)
    v = tf.random.uniform((), 0.0, 1.0 - s)
    img = tf.image.crop_and_resize(img[None], [[v, u, v + s, u + s]], [0], (IMG, IMG))[0]
    cx, cy = (boxes[:, 0] - u) / s, (boxes[:, 1] - v) / s
    x0 = tf.clip_by_value((boxes[:, 0] - boxes[:, 2] / 2 - u) / s, 0.0, 1.0)
    x1 = tf.clip_by_value((boxes[:, 0] + boxes[:, 2] / 2 - u) / s, 0.0, 1.0)
    y0 = tf.clip_by_value((boxes[:, 1] - boxes[:, 3] / 2 - v) / s, 0.0, 1.0)
    y1 = tf.clip_by_value((boxes[:, 1] + boxes[:, 3] / 2 - v) / s, 0.0, 1.0)
    keep = (cls >= 0) & (cx > 0) & (cx < 1) & (cy > 0) & (cy < 1) & (x1 - x0 > 0.01) & (y1 - y0 > 0.01)
    boxes = tf.stack([(x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0], -1)
    cls = tf.where(keep, cls, -1)

    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.7, 1.3)
    img = tf.clip_by_value(img, 0.0, 1.0) ** tf.random.uniform((), 0.7, 1.4)
    img = img + tf.random.normal(tf.shape(img), 0.0, tf.random.uniform((), 0.0, 0.04))
    return tf.clip_by_value(img, 0.0, 1.0), boxes, cls


def make_dataset(df, encoder, training, batch=64):
    gray, ir, B, C = pack(df)
    ds = tf.data.Dataset.from_tensor_slices((gray, ir, B, C))
    if training:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)

    def _map(g, i, b, c):
        if training and VARIANT == "mix":
            a = tf.random.uniform(())
            img = a * load_image(g) + (1.0 - a) * load_image(i)
        else:
            img = load_image(g if (VARIANT if training else EVAL_VARIANT) == "gray" else i)
        if training:
            img, b, c = augment(img, b, c)
        y = tf.numpy_function(encoder, [b, c], tf.float32)
        y.set_shape(encoder.shape)
        return img, y

    return ds.map(_map, num_parallel_calls=AUTOTUNE).batch(batch).prefetch(AUTOTUNE)


def image_dataset(df, variant=None, batch=128):
    gray, ir, _, _ = pack(df)
    paths_ = gray if (variant or EVAL_VARIANT) == "gray" else ir
    return tf.data.Dataset.from_tensor_slices(paths_).map(load_image, num_parallel_calls=AUTOTUNE).batch(batch)

## The grid detector: building blocks, target encoding, loss and decoding

This is the **modern tiny-YOLO head** used by step 2 of notebook 12 and by notebook 14. The
input image is covered by a `G × G` grid at stride 8, so a 160 px input gives a 20 × 20 grid
and each cell is 8 px. For every cell the network outputs `5 + C` numbers:

| channels | meaning | activation / decoding |
|---|---|---|
| `t_obj` | is an object centred near this cell? | `σ(t_obj)` |
| `t_x, t_y` | centre offset | `c_x = (j + 2σ(t_x) − 0.5) / G` (YOLOv5 form: reach ±1 cell) |
| `t_w, t_h` | size, as a fraction of the image | `w = σ(t_w)`, `h = σ(t_h)` (anchor-free) |
| `t_c1..t_cC` | class scores | `σ(t_c)` (independent, multi-label BCE) |

**Assignment.** A ground-truth box is positive in its centre cell **plus the nearest
horizontal and vertical neighbour** (YOLOv5's trick). That gives up to 3 positives per object,
which helps a lot with so few objects per image. When two boxes claim the same cell, the
smaller one wins (an eye is more specific than the head box around it).

**Loss**, normalised by the number of positive cells `N₊`:

$$\mathcal{L} = \lambda_{box}\frac{1}{N_+}\sum_{pos}(1-\mathrm{GIoU}(\hat b,b)) + \lambda_{obj}\frac{1}{N_+}\sum_{all}\mathrm{BCE}(\sigma(t_{obj}),\ \mathbb{1}_{pos}\cdot\mathrm{IoU}(\hat b,b)) + \lambda_{cls}\frac{1}{N_+}\sum_{pos}\sum_c \mathrm{BCE}(\sigma(t_c), y_c)$$

- The objectness target is the predicted box's IoU (YOLOv5), so confidence ranks well-localised
  boxes higher.
- GIoU still gives a gradient when boxes don't overlap.
- The objectness and class biases start at `log(0.01/0.99)` (RetinaNet prior), so the many
  empty cells don't swamp early training.

**Blocks**, kept int8- and CPU-friendly with ReLU6 activations (YOLO uses SiLU):
- `conv_bn`: conv, then batch norm, then ReLU6. Optionally depthwise-separable.
- `c2f_lite`: YOLOv8's C2f in miniature, i.e. a CSP split, a chain of small bottlenecks, then
  concatenating every intermediate.
- `sppf_lite`: three chained 5×5 max-pools, concatenated, which enlarge the receptive field
  cheaply. The head box covers most of the image.
- `decoupled_head`: separate small branches for classification/objectness and for box regression.

**Shared verbatim by notebooks 12 / 14.**

In [ ]:
import keras
from keras import layers

BN_MOMENTUM = 0.97      # YOLOv5 value; Keras default 0.99 lags too far behind fast-moving weights
PRIOR_BIAS = float(np.log(0.01 / 0.99))
GRID_STRIDE = 8
G = IMG // GRID_STRIDE


def conv_bn(x, c, k=3, s=1, dw=False):
    if dw and k > 1:
        x = layers.DepthwiseConv2D(k, s, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization(momentum=BN_MOMENTUM)(x)
        x = layers.ReLU(6.0)(x)
        x = layers.Conv2D(c, 1, use_bias=False)(x)
    else:
        x = layers.Conv2D(c, k, s, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization(momentum=BN_MOMENTUM)(x)
    return layers.ReLU(6.0)(x)


def c2f_lite(x, c, n=1):
    h = c // 2
    a, b = conv_bn(x, h, 1), conv_bn(x, h, 1)
    outs = [a, b]
    for _ in range(n):
        y = conv_bn(conv_bn(b, h, 3, dw=True), h, 3, dw=True)
        b = layers.Add()([b, y])
        outs.append(b)
    return conv_bn(layers.Concatenate()(outs), c, 1)


def sppf_lite(x, c):
    x = conv_bn(x, c // 2, 1)
    p1 = layers.MaxPooling2D(5, 1, padding="same")(x)
    p2 = layers.MaxPooling2D(5, 1, padding="same")(p1)
    p3 = layers.MaxPooling2D(5, 1, padding="same")(p2)
    return conv_bn(layers.Concatenate()([x, p1, p2, p3]), c, 1)


def decoupled_head(x, c, num_classes=NUM_CLASSES):
    prior = keras.initializers.Constant(PRIOR_BIAS)
    cls_branch = conv_bn(x, c, 3, dw=True)
    obj = layers.Conv2D(1, 1, bias_initializer=prior, name="obj")(cls_branch)
    cls = layers.Conv2D(num_classes, 1, bias_initializer=prior, name="cls")(cls_branch)
    box = layers.Conv2D(4, 1, name="box")(conv_bn(x, c, 3, dw=True))
    return layers.Concatenate(name="grid")([obj, box, cls])     # (G, G, 1 + 4 + C)


class GridEncoder:
    """NumPy target builder for the grid head: (G, G, 5 + C) = [obj, cx, cy, w, h, one-hot]."""

    def __init__(self, grid=G, num_classes=NUM_CLASSES, neighbours=True):
        self.grid, self.nc, self.neighbours = grid, num_classes, neighbours
        self.shape = (grid, grid, 5 + num_classes)

    def __call__(self, boxes, cls):
        t = np.zeros(self.shape, np.float32)
        valid = cls >= 0
        b, c = boxes[valid], cls[valid]
        for k in np.argsort(-(b[:, 2] * b[:, 3])):          # big first, small boxes overwrite
            cx, cy, w, h = b[k]
            gx, gy = cx * self.grid, cy * self.grid
            j, i = min(int(gx), self.grid - 1), min(int(gy), self.grid - 1)
            cells = [(i, j)]
            if self.neighbours:
                fx, fy = gx - j, gy - i
                if fx < 0.5 and j > 0:
                    cells.append((i, j - 1))
                elif fx >= 0.5 and j < self.grid - 1:
                    cells.append((i, j + 1))
                if fy < 0.5 and i > 0:
                    cells.append((i - 1, j))
                elif fy >= 0.5 and i < self.grid - 1:
                    cells.append((i + 1, j))
            for ii, jj in cells:
                t[ii, jj] = 0.0
                t[ii, jj, :5] = (1.0, cx, cy, w, h)
                t[ii, jj, 5 + int(c[k])] = 1.0
        return t


def _grid_xywh_tf(raw):
    g = tf.shape(raw)[1]
    j = tf.cast(tf.range(g), tf.float32)[None, None, :]
    i = tf.cast(tf.range(g), tf.float32)[None, :, None]
    gf = tf.cast(g, tf.float32)
    cx = (j + 2.0 * tf.sigmoid(raw[..., 1]) - 0.5) / gf
    cy = (i + 2.0 * tf.sigmoid(raw[..., 2]) - 0.5) / gf
    return cx, cy, tf.sigmoid(raw[..., 3]), tf.sigmoid(raw[..., 4])


def iou_giou_tf(a, b):
    """a, b: (..., 4) as cx, cy, w, h -> (IoU, GIoU)."""
    ax0, ay0, ax1, ay1 = a[..., 0] - a[..., 2] / 2, a[..., 1] - a[..., 3] / 2, a[..., 0] + a[..., 2] / 2, a[..., 1] + a[..., 3] / 2
    bx0, by0, bx1, by1 = b[..., 0] - b[..., 2] / 2, b[..., 1] - b[..., 3] / 2, b[..., 0] + b[..., 2] / 2, b[..., 1] + b[..., 3] / 2
    inter = tf.nn.relu(tf.minimum(ax1, bx1) - tf.maximum(ax0, bx0)) * tf.nn.relu(tf.minimum(ay1, by1) - tf.maximum(ay0, by0))
    union = a[..., 2] * a[..., 3] + b[..., 2] * b[..., 3] - inter + 1e-9
    iou = inter / union
    hull = (tf.maximum(ax1, bx1) - tf.minimum(ax0, bx0)) * (tf.maximum(ay1, by1) - tf.minimum(ay0, by0)) + 1e-9
    return iou, iou - (hull - union) / hull


def grid_loss(lam_box=5.0, lam_obj=1.0, lam_cls=1.0):
    def loss(y_true, y_pred):
        pos = y_true[..., 0]
        npos = tf.maximum(tf.reduce_sum(pos), 1.0)
        pred = tf.stack(_grid_xywh_tf(y_pred), -1)
        iou, giou = iou_giou_tf(pred, y_true[..., 1:5])
        box = tf.reduce_sum((1.0 - giou) * pos) / npos
        obj_t = pos * tf.stop_gradient(tf.clip_by_value(iou, 0.0, 1.0))
        obj = tf.reduce_sum(tf.nn.sigmoid_cross_entropy_with_logits(labels=obj_t, logits=y_pred[..., 0])) / npos
        cls = tf.reduce_sum(tf.reduce_sum(tf.nn.sigmoid_cross_entropy_with_logits(
            labels=y_true[..., 5:], logits=y_pred[..., 5:]), -1) * pos) / npos
        return lam_box * box + lam_obj * obj + lam_cls * cls
    return loss


def _sig(x):
    return 1.0 / (1.0 + np.exp(-x))


def grid_postprocess(raw, conf_floor=0.001, pre_nms=200):
    """One image's raw grid (G, G, 5 + C) -> (xyxy boxes, scores, classes) after NMS."""
    g = raw.shape[0]
    j, i = np.meshgrid(np.arange(g), np.arange(g))
    cx = (j + 2 * _sig(raw[..., 1]) - 0.5) / g
    cy = (i + 2 * _sig(raw[..., 2]) - 0.5) / g
    w, h = _sig(raw[..., 3]), _sig(raw[..., 4])
    cls_p = _sig(raw[..., 5:])
    cls = cls_p.argmax(-1)
    score = _sig(raw[..., 0]) * cls_p.max(-1)
    m = score > conf_floor
    order = np.argsort(-score[m])[:pre_nms]
    boxes = np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], -1)[m][order].clip(0, 1)
    return nms_per_class(boxes, score[m][order], cls[m][order])

### Training, prediction and int8 export (shared by every Keras detector here)

- `ValMAP` computes the validation mAP@0.5 every few epochs and exposes it as `val_map50` for
  early stopping and best-weight selection. Loss alone is a poor proxy for detection quality.
- `export_tflite_int8` does full-integer quantisation calibrated on 200 training images. Input
  and output stay float32, and everything inside is int8. That's what the Pi runs, through
  XNNPACK.

In [ ]:
class ValMAP(keras.callbacks.Callback):
    def __init__(self, df, decode, every=5, max_images=1500):
        super().__init__()
        self.df = df.sample(min(len(df), max_images), random_state=SEED) if len(df) > max_images else df
        self.decode, self.every, self.best, self.last, self.best_weights = decode, every, -1.0, 0.0, None

    def on_epoch_end(self, epoch, logs=None):
        if epoch % self.every == 0 or epoch == self.params["epochs"] - 1:
            preds = predict_keras(self.model, self.df, self.decode)
            gts = gts_for(self.df)
            self.last = float(np.nanmean([average_precision(preds, gts, c, [0.5])[0] for c in range(NUM_CLASSES)]))
            if self.last > self.best:
                self.best, self.best_weights = self.last, self.model.get_weights()
            print(f"  val mAP50 = {self.last:.4f} (best {self.best:.4f})")
        logs["val_map50"] = self.last


def train_detector(model, loss, encoder, decode, epochs, lr=2e-3, batch=64, patience=6, every=5):
    epochs = 2 if SMOKE else epochs
    train_ds = make_dataset(splits["train"], encoder, training=True, batch=batch)
    steps = max(1, int(np.ceil(len(splits["train"]) / batch)))
    sched = keras.optimizers.schedules.CosineDecay(lr, decay_steps=epochs * steps, alpha=0.05)
    model.compile(optimizer=keras.optimizers.AdamW(sched, weight_decay=5e-4), loss=loss)
    vm = ValMAP(splits["valid"], decode, every=1 if SMOKE else every)
    stop = keras.callbacks.EarlyStopping(monitor="val_map50", mode="max", patience=patience * every,
                                         restore_best_weights=False)
    hist = model.fit(train_ds, epochs=epochs, callbacks=[vm, stop], verbose=2)
    if vm.best_weights is not None:
        model.set_weights(vm.best_weights)
    recalibrate_bn(model, encoder)
    return hist, vm.best


def recalibrate_bn(model, encoder, n_batches=100, batch=64):
    """Refresh BatchNorm running statistics for the FINAL weights: forward passes in training
    mode over clean (un-augmented) train images, with weights untouched. Without this, the
    running averages lag behind weights that were still moving, and eval-mode outputs drift. A
    memorisation test showed that this alone took eval mAP@0.5 from 0.00 to 0.99 for the same
    weights."""
    ds = make_dataset(splits["train"], encoder, training=False, batch=batch).take(2 if SMOKE else n_batches)
    for x, _ in ds:
        model(x, training=True)


def predict_keras(model, df, decode, variant=None, batch=128):
    raw = model.predict(image_dataset(df, variant, batch), verbose=0)
    return {iid: decode(r) for iid, r in zip(df.image_id, raw)}


def export_tflite_int8(model, path, n_calib=200):
    sm_dir = f"{WORK_DIR}/saved_model_{model.name}"
    if hasattr(model, "export"):
        model.export(sm_dir, verbose=False)
    else:
        tf.saved_model.save(model, sm_dir)
    conv = tf.lite.TFLiteConverter.from_saved_model(sm_dir)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    calib = splits["train"].sample(min(n_calib, len(splits["train"])), random_state=SEED)
    imgs = [x.numpy() for x in image_dataset(calib, batch=1)]
    conv.representative_dataset = lambda: ([x] for x in imgs)
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    with open(path, "wb") as f:
        f.write(conv.convert())
    print(f"-> {path} ({os.path.getsize(path) / 1024:.1f} KB)")
    return path


def predict_tflite(path, df, decode, variant=None):
    it = tf.lite.Interpreter(model_path=str(path), num_threads=4)
    it.allocate_tensors()
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    preds = {}
    for iid, x in zip(df.image_id, image_dataset(df, variant, batch=1)):
        it.set_tensor(inp["index"], x.numpy().astype(np.float32)); it.invoke()
        preds[iid] = decode(it.get_tensor(out["index"])[0])
    return preds

## Step 1: YOLOv1-style baseline

This follows Redmon et al. (2016) at a tiny scale. The image is divided into an `S × S` grid
(`S = 10`, so stride 16 and 16 px cells at 160 px). The cell containing an object's centre is
**responsible** for it and predicts:

- `C`: confidence that an object is centred here, with the box IoU as its target;
- `x, y`: the centre **offset inside the cell** (0–1, sigmoid);
- `√w, √h`: square-rooted width and height (sigmoid). The square root makes an error on a small
  eye cost more than the same error on the large head box;
- `p(class)`: a softmax over the 5 classes.

The loss is the original sum of squared errors:

$$\mathcal{L}=\lambda_{coord}\sum_{obj}\big[(x-\hat x)^2+(y-\hat y)^2+(\sqrt w-\sqrt{\hat w})^2+(\sqrt h-\sqrt{\hat h})^2\big] + \sum_{obj}(C-\hat C)^2 + \lambda_{noobj}\sum_{noobj}\hat C^2 + \sum_{obj}\sum_c (p_c-\hat p_c)^2$$

with `λ_coord = 5` and `λ_noobj = 0.5`. One box per cell means two objects centred in the same
cell can't both be found; the smaller one wins at encoding time. The network is the "tiny-YOLO"
pattern of conv + max-pool stages with leaky ReLU.

In [ ]:
S1 = IMG // 16


def build_yolov1_tiny(width=1.0, name="yolov1_tiny"):
    c = lambda n: max(8, int(round(n * width / 8)) * 8)
    inp = keras.Input((IMG, IMG, 1))
    x = inp
    for n in (16, 32, 64, 128):                      # 160 -> 80 -> 40 -> 20 -> 10
        x = layers.Conv2D(c(n), 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization(momentum=BN_MOMENTUM)(x)
        x = layers.LeakyReLU(0.1)(x)
        x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(c(128), 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization(momentum=BN_MOMENTUM)(x)
    x = layers.LeakyReLU(0.1)(x)
    out = layers.Conv2D(5 + NUM_CLASSES, 1, name="yolo_v1_grid")(x)   # (S, S, 5 + C), raw
    return keras.Model(inp, out, name=name)


class V1Encoder:
    """[conf, x_off, y_off, sqrt_w, sqrt_h, one-hot]; the centre cell only, smaller box wins."""

    def __init__(self, s=S1):
        self.s, self.shape = s, (s, s, 5 + NUM_CLASSES)

    def __call__(self, boxes, cls):
        t = np.zeros(self.shape, np.float32)
        valid = cls >= 0
        b, c = boxes[valid], cls[valid]
        for k in np.argsort(-(b[:, 2] * b[:, 3])):
            cx, cy, w, h = b[k]
            j, i = min(int(cx * self.s), self.s - 1), min(int(cy * self.s), self.s - 1)
            t[i, j] = 0.0
            t[i, j, :5] = (1.0, cx * self.s - j, cy * self.s - i, np.sqrt(w), np.sqrt(h))
            t[i, j, 5 + int(c[k])] = 1.0
        return t


def yolov1_loss(l_coord=5.0, l_noobj=0.5):
    def loss(y_true, y_pred):
        s = tf.cast(tf.shape(y_pred)[1], tf.float32)
        obj = y_true[..., 0]
        p_conf = tf.sigmoid(y_pred[..., 0])
        p_xy, p_wh = tf.sigmoid(y_pred[..., 1:3]), tf.sigmoid(y_pred[..., 3:5])
        p_cls = tf.nn.softmax(y_pred[..., 5:])
        t_xy, t_wh = y_true[..., 1:3], y_true[..., 3:5]
        # IoU between predicted and true box (cell-relative centres share the cell) = conf target
        pb = tf.concat([p_xy / s, p_wh ** 2], -1)
        tb = tf.concat([t_xy / s, t_wh ** 2], -1)
        iou, _ = iou_giou_tf(pb, tb)
        coord = tf.reduce_sum(obj * (tf.reduce_sum((p_xy - t_xy) ** 2, -1) + tf.reduce_sum((p_wh - t_wh) ** 2, -1)))
        conf_obj = tf.reduce_sum(obj * (p_conf - tf.stop_gradient(iou)) ** 2)
        conf_noobj = tf.reduce_sum((1.0 - obj) * p_conf ** 2)
        cls_err = tf.reduce_sum(obj * tf.reduce_sum((p_cls - y_true[..., 5:]) ** 2, -1))
        batch = tf.cast(tf.shape(y_pred)[0], tf.float32)
        return (l_coord * coord + conf_obj + l_noobj * conf_noobj + cls_err) / batch
    return loss


def v1_postprocess(raw, conf_floor=0.001):
    s = raw.shape[0]
    j, i = np.meshgrid(np.arange(s), np.arange(s))
    cx = (j + _sig(raw[..., 1])) / s
    cy = (i + _sig(raw[..., 2])) / s
    w, h = _sig(raw[..., 3]) ** 2, _sig(raw[..., 4]) ** 2
    e = np.exp(raw[..., 5:] - raw[..., 5:].max(-1, keepdims=True))
    p = e / e.sum(-1, keepdims=True)
    score = _sig(raw[..., 0]) * p.max(-1)
    m = score > conf_floor
    boxes = np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], -1)[m].clip(0, 1)
    return nms_per_class(boxes, score[m], p.argmax(-1)[m])

Training step 1. A sanity check first: **overfit one batch.** If the loss can't be driven
close to zero on 16 images, the encoding, loss or decoding is wrong, and there's no point
spending GPU time on the full run.

In [ ]:
def overfit_check(model, loss, encoder, steps=150):
    batch = next(iter(make_dataset(splits["train"].head(16), encoder, training=False, batch=16)))
    model.compile(optimizer=keras.optimizers.Adam(3e-3), loss=loss)
    first = float(model.train_on_batch(*batch))
    for _ in range(30 if SMOKE else steps):
        last = float(model.train_on_batch(*batch))
    print(f"overfit one batch: loss {first:.3f} -> {last:.3f}")
    return first, last

In [ ]:
overfit_check(build_yolov1_tiny(), yolov1_loss(), V1Encoder())

v1 = build_yolov1_tiny()
print(f"{v1.name}: {v1.count_params():,} params")
v1_hist, v1_best = train_detector(v1, yolov1_loss(), V1Encoder(), v1_postprocess, epochs=80)
v1_val = report(predict_keras(v1, splits["valid"], v1_postprocess), splits["valid"], "YOLOv1-tiny / valid")

## Step 2: Modern tiny YOLO (CSP + SPPF + decoupled, anchor-free head)

Same input and the same data. The architecture follows YOLOv5 and YOLOv8, scaled down:

    160×160×1 ─ stem conv s2 ─ dw-conv s2 ─ C2f-lite ─ dw-conv s2 ─ C2f-lite×2 ─ SPPF-lite ─ decoupled head
       160         80            40          40          20            20            20        20×20×(5+C)

The changes relative to step 1:
- a finer grid (stride 8, so 20 × 20). Eyes are only about 25 px, so a 16 px cell is coarse for
  them;
- neighbour-cell assignment, giving 3 positives per object;
- IoU-aware objectness;
- GIoU box loss;
- sigmoid multi-label classes.

`width` scales every channel count; it's the main size knob.

In [ ]:
def build_tiny_yolo(width=1.0, name=None):
    c = lambda n: max(8, int(round(n * width / 8)) * 8)
    inp = keras.Input((IMG, IMG, 1))
    x = conv_bn(inp, c(16), 3, 2)              # 80
    x = conv_bn(x, c(32), 3, 2, dw=True)       # 40
    x = c2f_lite(x, c(32), 1)
    x = conv_bn(x, c(64), 3, 2, dw=True)       # 20 (stride 8)
    x = c2f_lite(x, c(64), 2)
    x = sppf_lite(x, c(64))
    out = decoupled_head(x, c(48))
    return keras.Model(inp, out, name=name or f"tiny_yolo_w{width:g}")

In [ ]:
overfit_check(build_tiny_yolo(), grid_loss(), GridEncoder())

WIDTH = 1.0
ty = build_tiny_yolo(WIDTH)
print(f"{ty.name}: {ty.count_params():,} params")
ty_hist, ty_best = train_detector(ty, grid_loss(), GridEncoder(), grid_postprocess, epochs=150)
ty_val = report(predict_keras(ty, splits["valid"], grid_postprocess), splits["valid"], f"{ty.name} / valid")

## Picking the winner, then test-set evaluation (float and int8)

The step with the higher **validation** mAP@0.5 is kept. The test set is touched exactly once,
for the kept model only. The int8 TFLite model is evaluated too: quantisation must not cost
much accuracy, or the latency win is meaningless.

In [ ]:
candidates = {"yolov1_tiny": (v1, v1_postprocess, v1_val["all"]["map50"]),
              ty.name: (ty, grid_postprocess, ty_val["all"]["map50"])}
print({k: round(v[2], 4) for k, v in candidates.items()})
best_name = max(candidates, key=lambda k: candidates[k][2])
best_model, best_decode, _ = candidates[best_name]
print(f"kept: {best_name}")

keras_path = f"{models_folder}/{best_name}_{VERSION_STR}.keras"
best_model.save(keras_path)
test_float = report(predict_keras(best_model, splits["test"], best_decode), splits["test"], f"{best_name} / TEST float")
tflite_path = export_tflite_int8(best_model, f"{models_folder}/{best_name}_{VERSION_STR}_int8.tflite")
test_int8 = report(predict_tflite(tflite_path, splits["test"], best_decode), splits["test"], f"{best_name} / TEST int8")
latency = bench_tflite(tflite_path)

save_metrics(
    best_name, "12_tiny_grid_detector",
    params=int(best_model.count_params()), int8_kb=round(os.path.getsize(tflite_path) / 1024, 1),
    val_map50={k: v[2] for k, v in candidates.items()},
    test_float=test_float, test_int8=test_int8, latency_ms=latency,
    keras_path=keras_path, tflite_path=tflite_path)

## Optional ablations

These are off by default because they cost GPU time. Each one retrains step 2 and saves its own
metrics file, so notebook 14's table picks it up:

- **width** ×0.5 / ×2: the accuracy-vs-latency curve;
- **variant** `gray` vs `ir` vs `mix`: which training images transfer best to the IR camera;
- **no close-ups**: does MRL actually help face-crop eyes? Set `INCLUDE_CLOSEUPS = False` in
  Setup and rerun from the Loading cell.

In [ ]:
RUN_ABLATIONS = False
if RUN_ABLATIONS:
    for w in (0.5, 2.0):
        m = build_tiny_yolo(w)
        train_detector(m, grid_loss(), GridEncoder(), grid_postprocess, epochs=150)
        p = export_tflite_int8(m, f"{models_folder}/{m.name}_{VERSION_STR}_int8.tflite")
        save_metrics(m.name, "12_tiny_grid_detector", params=int(m.count_params()),
                     int8_kb=round(os.path.getsize(p) / 1024, 1),
                     test_float=report(predict_keras(m, splits["test"], grid_postprocess), splits["test"], m.name),
                     test_int8=report(predict_tflite(p, splits["test"], grid_postprocess), splits["test"], m.name + " int8"),
                     latency_ms=bench_tflite(p), tflite_path=p)

## Download the results (only when not saving to Drive)

With `USE_DRIVE = False`, the models and metrics live only on this session's disk and vanish
when it ends. This cell zips `models/parts_det/` and downloads it to your computer. With Drive
on, it does nothing, because the files are already in `MyDrive/Argus/models/parts_det/`.

**Shared verbatim by notebooks 12 / 13 / 14.**

In [ ]:
if IN_COLAB and not USE_DRIVE and not SMOKE:
    from google.colab import files
    archive = shutil.make_archive(f"{WORK_DIR}/parts_det_results_{VERSION_STR}", "zip", models_folder)
    files.download(archive)
else:
    print(f"results are in {models_folder}")